## Task 1: Naive Bayes Classification on Titanic Data

In [16]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

titanic = fetch_openml(data_id=40945, as_frame=True, parser='auto')
df = titanic.frame

features = ['pclass', 'sex', 'embarked']
target = 'survived'
df_selected = df[features + [target]]
print(df_selected.isna().sum())
df_clean = df_selected.dropna()
print(f"Original dataset shape: {df.shape}")
print(f"Cleaned dataset shape: {df_clean.shape}")

pclass      0
sex         0
embarked    2
survived    0
dtype: int64
Original dataset shape: (1309, 14)
Cleaned dataset shape: (1307, 4)


In [17]:
X = df_clean[features]
y = df_clean[target].astype(int) # Ensure target is integer type

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (914, 3)
Testing set shape: (393, 3)


In [35]:
class_counts = y_train.value_counts()
print(class_counts)
total_train = len(y_train)
class_priors = {c: count / total_train for c, count in class_counts.items()}
print(class_priors)
conditional_probs = {}

for feature in features:
    conditional_probs[feature] = {}
    unique_categories = df_clean[feature].unique()
    num_unique = len(unique_categories)
    for c in class_priors.keys():
        conditional_probs[feature][c] = {}
        X_train_class = X_train[y_train == c]
        feature_counts = X_train_class[feature].value_counts()
        total_class_samples = len(X_train_class)
        
        for category in unique_categories:
            count = feature_counts.get(category, 0)
            prob = (count + 1) / (total_class_samples + num_unique)
            conditional_probs[feature][c][category] = prob
            

survived
0    566
1    348
Name: count, dtype: int64
{0: 0.6192560175054704, 1: 0.38074398249452956}


In [33]:
def predict_naive_bayes(X_data, priors, conditionals):
    predictions = []
    log_scores_list = []
    
    for _, row in X_data.iterrows():
        class_scores = {}
        for c in priors.keys():
            # Start with log of the prior: log P(Y=c)
            score = np.log(priors[c])
            
            # Add log of conditional probabilities: sum(log P(X_j = x_j | Y=c))
            for feature in features:
                val = row[feature]
                # Default to a very small probability if a category was never seen (fallback)
                prob = conditionals[feature][c].get(val, 1e-6)
                score += np.log(prob)
                
            class_scores[c] = score
            
        predicted_class = max(class_scores, key=class_scores.get)
        predictions.append(predicted_class)
        log_scores_list.append(class_scores)
        
    return np.array(predictions), log_scores_list

In [34]:
for c, prob in class_priors.items():
    print(f"Class {c} (Survived={c}): {prob:.4f}")

for feature in features:
    print(f"\nFeature: {feature.upper()}")
    cond_df = pd.DataFrame(conditional_probs[feature])
    cond_df.columns = [f"P(X|Y={c})" for c in cond_df.columns]
    print(cond_df)

Class 0 (Survived=0): 0.6193
Class 1 (Survived=1): 0.3807

Feature: PCLASS
   P(X|Y=0)  P(X|Y=1)
1  0.165202  0.381766
2  0.189807  0.236467
3  0.644991  0.381766

Feature: SEX
        P(X|Y=0)  P(X|Y=1)
female   0.15669  0.665714
male     0.84331  0.334286

Feature: EMBARKED
   P(X|Y=0)  P(X|Y=1)
S  0.741652  0.598291
C  0.147627  0.307692
Q  0.110721  0.094017


In [31]:
y_pred, test_log_scores = predict_naive_bayes(X_test, class_priors, conditional_probs)

accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Test Accuracy: {accuracy * 100:.2f}%")
print("\nConfusion Matrix:")
print("                 Predicted Class 0 | Predicted Class 1")
print(f"Actual Class 0 | {conf_matrix[0, 0]:^17} | {conf_matrix[0, 1]:^17}")
print(f"Actual Class 1 | {conf_matrix[1, 0]:^17} | {conf_matrix[1, 1]:^17}")

Test Accuracy: 77.61%

Confusion Matrix:
                 Predicted Class 0 | Predicted Class 1
Actual Class 0 |        194        |        49        
Actual Class 1 |        39         |        111       


In [36]:
np.random.seed(42) 
sample_indices = np.random.choice(len(X_test), 5, replace=False)

for idx in sample_indices:
    passenger = X_test.iloc[idx]
    true_class = y_test.iloc[idx]
    predicted = y_pred[idx]
    scores = test_log_scores[idx]
    
    print(f"\nPassenger Test Index {idx}:")
    print(f"  Attributes: pclass={passenger['pclass']}, sex={passenger['sex']}, embarked={passenger['embarked']}")
    print(f"  True Class: {true_class}")
    print(f"  Log-Scores: S(0) = {scores[0]:.4f}, S(1) = {scores[1]:.4f}")
    print(f"  Predicted Class: {predicted}")


Passenger Test Index 78:
  Attributes: pclass=2, sex=male, embarked=S
  True Class: 0
  Log-Scores: S(0) = -2.6103, S(1) = -4.0170
  Predicted Class: 0

Passenger Test Index 275:
  Attributes: pclass=2, sex=male, embarked=S
  True Class: 0
  Log-Scores: S(0) = -2.6103, S(1) = -4.0170
  Predicted Class: 0

Passenger Test Index 247:
  Attributes: pclass=1, sex=male, embarked=C
  True Class: 0
  Log-Scores: S(0) = -4.3633, S(1) = -4.2030
  Predicted Class: 1

Passenger Test Index 55:
  Attributes: pclass=2, sex=male, embarked=S
  True Class: 1
  Log-Scores: S(0) = -2.6103, S(1) = -4.0170
  Predicted Class: 0

Passenger Test Index 388:
  Attributes: pclass=1, sex=male, embarked=C
  True Class: 0
  Log-Scores: S(0) = -4.3633, S(1) = -4.2030
  Predicted Class: 1
